<a href="https://colab.research.google.com/github/VICO-27/CSEC-DS-Summer/blob/main/GeoAI_Feature_Eng_and_Mode%3Bl_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup + Load

In [20]:
# ============================================================
#  Setup + Load
# ============================================================

import pandas as pd
import numpy as np

# >>> EDIT THIS PATH to wherever you put the files in Drive <
DATA_DIR = '/content/drive/MyDrive'

train = pd.read_csv(f'{DATA_DIR}/Train.csv')
test  = pd.read_csv(f'{DATA_DIR}/Test.csv')
ss    = pd.read_csv(f'{DATA_DIR}/SampleSubmission.csv')

print("TRAIN shape:", train.shape)
print("TEST shape :", test.shape)
print("SAMPLE SUB shape:", ss.shape)
print()
print("TRAIN columns:")
print(train.columns.tolist())
print()
print("SAMPLE SUB head:")
print(ss.head())

TRAIN shape: (1821, 146)
TEST shape : (1030, 145)
SAMPLE SUB shape: (1030, 3)

TRAIN columns:
['ID', 'label', 'VH_01', 'VV_01', 'blue_01', 'green_01', 'nir_01', 'nira_01', 're1_01', 're2_01', 're3_01', 'red_01', 'swir1_01', 'swir2_01', 'VH_02', 'VV_02', 'blue_02', 'green_02', 'nir_02', 'nira_02', 're1_02', 're2_02', 're3_02', 'red_02', 'swir1_02', 'swir2_02', 'VH_03', 'VV_03', 'blue_03', 'green_03', 'nir_03', 'nira_03', 're1_03', 're2_03', 're3_03', 'red_03', 'swir1_03', 'swir2_03', 'VH_04', 'VV_04', 'blue_04', 'green_04', 'nir_04', 'nira_04', 're1_04', 're2_04', 're3_04', 'red_04', 'swir1_04', 'swir2_04', 'VH_05', 'VV_05', 'blue_05', 'green_05', 'nir_05', 'nira_05', 're1_05', 're2_05', 're3_05', 'red_05', 'swir1_05', 'swir2_05', 'VH_06', 'VV_06', 'blue_06', 'green_06', 'nir_06', 'nira_06', 're1_06', 're2_06', 're3_06', 'red_06', 'swir1_06', 'swir2_06', 'VH_07', 'VV_07', 'blue_07', 'green_07', 'nir_07', 'nira_07', 're1_07', 're2_07', 're3_07', 'red_07', 'swir1_07', 'swir2_07', 'VH_08',

#  Identify ID / target / feature columns, band-month structure


In [21]:
# ============================================================
#  Identify ID / target / feature columns, band-month structure
# ============================================================
id_col = train.columns[0]  # confirm this is the ID col
target_candidates = [c for c in train.columns if c not in test.columns]
print("Likely target column(s):", target_candidates)

feature_cols = [c for c in test.columns if c != id_col]
print("Num feature cols:", len(feature_cols))
print(feature_cols[:30])

# Try to parse band + month out of column names (adjust parsing once we see real names)
import re
parsed = []
for c in feature_cols:
    parsed.append(c)
print("\nUnique-looking tokens (first 50 cols):")
for c in feature_cols[:50]:
    print(c)

Likely target column(s): ['label']
Num feature cols: 144
['VH_01', 'VV_01', 'blue_01', 'green_01', 'nir_01', 'nira_01', 're1_01', 're2_01', 're3_01', 'red_01', 'swir1_01', 'swir2_01', 'VH_02', 'VV_02', 'blue_02', 'green_02', 'nir_02', 'nira_02', 're1_02', 're2_02', 're3_02', 'red_02', 'swir1_02', 'swir2_02', 'VH_03', 'VV_03', 'blue_03', 'green_03', 'nir_03', 'nira_03']

Unique-looking tokens (first 50 cols):
VH_01
VV_01
blue_01
green_01
nir_01
nira_01
re1_01
re2_01
re3_01
red_01
swir1_01
swir2_01
VH_02
VV_02
blue_02
green_02
nir_02
nira_02
re1_02
re2_02
re3_02
red_02
swir1_02
swir2_02
VH_03
VV_03
blue_03
green_03
nir_03
nira_03
re1_03
re2_03
re3_03
red_03
swir1_03
swir2_03
VH_04
VV_04
blue_04
green_04
nir_04
nira_04
re1_04
re2_04
re3_04
red_04
swir1_04
swir2_04
VH_05
VV_05


# Target distribution


In [22]:
# ============================================================
# Target distribution
# ============================================================
target_col = target_candidates[0] if target_candidates else None
print("Target column guess:", target_col)
if target_col:
    print(train[target_col].value_counts())
    print(train[target_col].value_counts(normalize=True))

Target column guess: label
label
0    1086
1     735
Name: count, dtype: int64
label
0    0.596376
1    0.403624
Name: proportion, dtype: float64


# Missingness structure (-9999)


In [23]:
# ============================================================
# Missingness structure (-9999)
# ============================================================
NA_VAL = -9999

# Overall fraction of -9999 per column
na_frac = (train[feature_cols] == NA_VAL).mean().sort_values(ascending=False)
print("Top 20 columns by % missing (-9999) in TRAIN:")
print(na_frac.head(20))

na_frac_test = (test[feature_cols] == NA_VAL).mean().sort_values(ascending=False)
print("\nTop 20 columns by % missing (-9999) in TEST:")
print(na_frac_test.head(20))

# Per-row: how many valid (non -9999) feature values does each row have?
train_valid_count = (train[feature_cols] != NA_VAL).sum(axis=1)
test_valid_count  = (test[feature_cols]  != NA_VAL).sum(axis=1)

print("\nTRAIN valid-value count per row — describe:")
print(train_valid_count.describe())
print("\nTEST valid-value count per row — describe:")
print(test_valid_count.describe())

Top 20 columns by % missing (-9999) in TRAIN:
VH_01       0.0
VV_01       0.0
blue_01     0.0
green_01    0.0
nir_01      0.0
nira_01     0.0
re1_01      0.0
re2_01      0.0
re3_01      0.0
red_01      0.0
swir1_01    0.0
swir2_01    0.0
VH_02       0.0
VV_02       0.0
blue_02     0.0
green_02    0.0
nir_02      0.0
nira_02     0.0
re1_02      0.0
re2_02      0.0
dtype: float64

Top 20 columns by % missing (-9999) in TEST:
green_01    0.875728
blue_01     0.875728
nira_01     0.875728
nir_01      0.875728
re1_01      0.875728
re2_01      0.875728
red_01      0.875728
re3_01      0.875728
swir1_01    0.875728
swir2_01    0.875728
VV_01       0.874757
VH_01       0.874757
green_12    0.872816
blue_12     0.872816
VV_12       0.872816
VH_12       0.872816
swir2_12    0.872816
swir1_12    0.872816
red_12      0.872816
re3_12      0.872816
dtype: float64

TRAIN valid-value count per row — describe:
count    1821.0
mean      144.0
std         0.0
min       144.0
25%       144.0
50%       144

#  Per-month completeness (needs month parsed from col name)


In [24]:
# ============================================================
#  Per-month completeness (needs month parsed from col name)
# ============================================================
# PLACEHOLDER: once we see actual column naming convention (e.g. "S2_B02_month3"
# or "VV_m1" etc.), I'll write precise regex to group columns by month & sensor.
# For now, print a wide sample of names so we can design the parser correctly:
print(feature_cols)

['VH_01', 'VV_01', 'blue_01', 'green_01', 'nir_01', 'nira_01', 're1_01', 're2_01', 're3_01', 'red_01', 'swir1_01', 'swir2_01', 'VH_02', 'VV_02', 'blue_02', 'green_02', 'nir_02', 'nira_02', 're1_02', 're2_02', 're3_02', 'red_02', 'swir1_02', 'swir2_02', 'VH_03', 'VV_03', 'blue_03', 'green_03', 'nir_03', 'nira_03', 're1_03', 're2_03', 're3_03', 'red_03', 'swir1_03', 'swir2_03', 'VH_04', 'VV_04', 'blue_04', 'green_04', 'nir_04', 'nira_04', 're1_04', 're2_04', 're3_04', 'red_04', 'swir1_04', 'swir2_04', 'VH_05', 'VV_05', 'blue_05', 'green_05', 'nir_05', 'nira_05', 're1_05', 're2_05', 're3_05', 'red_05', 'swir1_05', 'swir2_05', 'VH_06', 'VV_06', 'blue_06', 'green_06', 'nir_06', 'nira_06', 're1_06', 're2_06', 're3_06', 'red_06', 'swir1_06', 'swir2_06', 'VH_07', 'VV_07', 'blue_07', 'green_07', 'nir_07', 'nira_07', 're1_07', 're2_07', 're3_07', 'red_07', 'swir1_07', 'swir2_07', 'VH_08', 'VV_08', 'blue_08', 'green_08', 'nir_08', 'nira_08', 're1_08', 're2_08', 're3_08', 'red_08', 'swir1_08', 'sw

#  Understand TEST masking structure precisely


In [25]:
# ============================================================
#  Understand TEST masking structure precisely
# ============================================================
NA_VAL = -9999
months = [f"{m:02d}" for m in range(1, 13)]
bands = ['VH','VV','blue','green','nir','nira','re1','re2','re3','red','swir1','swir2']
s1_bands = ['VH','VV']
s2_bands = [b for b in bands if b not in s1_bands]

# For each row, which months have ANY valid data (i.e. not fully -9999 for that month)?
def month_validity_matrix(df):
    mat = pd.DataFrame(index=df.index)
    for m in months:
        cols_m = [f"{b}_{m}" for b in bands]
        mat[m] = (df[cols_m] != NA_VAL).any(axis=1)
    return mat

test_month_valid = month_validity_matrix(test)
n_active_months = test_month_valid.sum(axis=1)
print("Distribution of # active months per TEST row:")
print(n_active_months.value_counts().sort_index())

# Are active months always consecutive?
def is_consecutive(row):
    active_idx = [i for i, v in enumerate(row) if v]
    if len(active_idx) <= 1:
        return True
    return active_idx == list(range(active_idx[0], active_idx[0] + len(active_idx)))

consec_check = test_month_valid.apply(lambda r: is_consecutive(r.values), axis=1)
print("\nFraction of TEST rows with consecutive active-month window:", consec_check.mean())

# Within an "active" month, is S1 sometimes valid while S2 is fully missing?
mismatch_count = 0
total_checked = 0
for m in months:
    s1_cols = [f"{b}_{m}" for b in s1_bands]
    s2_cols = [f"{b}_{m}" for b in s2_bands]
    s1_valid = (test[s1_cols] != NA_VAL).any(axis=1)
    s2_valid = (test[s2_cols] != NA_VAL).any(axis=1)
    mismatch = (s1_valid & ~s2_valid)
    mismatch_count += mismatch.sum()
    total_checked += len(test)
print(f"\nRows where S1 valid but ALL S2 bands missing in same month (summed across months): {mismatch_count}")

# Check partial S2 missingness too (maybe some S2 bands missing, not all)
partial_s2 = 0
for m in months:
    s2_cols = [f"{b}_{m}" for b in s2_bands]
    valid_counts = (test[s2_cols] != NA_VAL).sum(axis=1)
    partial = ((valid_counts > 0) & (valid_counts < len(s2_cols))).sum()
    partial_s2 += partial
print(f"Rows with PARTIAL (some but not all) S2 bands missing within a month (summed across months): {partial_s2}")

# Where does the active window start? (helps understand if certain seasons are overrepresented in test)
first_active = test_month_valid.apply(lambda r: r.values.argmax() if r.any() else -1, axis=1)
print("\nDistribution of FIRST active month index (0=Jan):")
print(first_active.value_counts().sort_index())

Distribution of # active months per TEST row:
4    345
5    343
6    342
Name: count, dtype: int64

Fraction of TEST rows with consecutive active-month window: 1.0

Rows where S1 valid but ALL S2 bands missing in same month (summed across months): 320
Rows with PARTIAL (some but not all) S2 bands missing within a month (summed across months): 0

Distribution of FIRST active month index (0=Jan):
0    129
1    131
2    130
3    129
4    130
5    130
6    130
7     82
8     39
Name: count, dtype: int64


#  Sanity check label balance vs nothing else yet, and dtypes


In [26]:
# ============================================================
#  Sanity check label balance vs nothing else yet, and dtypes
# ============================================================
print(train['label'].dtype)
print(train[feature_cols].dtypes.value_counts())
print(train[feature_cols].describe().T[['min','max','mean']].head(20))

int64
int64      120
float64     24
Name: count, dtype: int64
                  min          max         mean
VH_01      -48.143182   -10.404720   -25.379836
VV_01      -31.912804     6.227268   -15.136799
blue_01   1070.000000  3070.000000  1651.551345
green_01  1094.000000  3788.000000  1845.806150
nir_01    1011.000000  6722.000000  2317.796266
nira_01   1005.000000  6528.000000  2281.221856
re1_01    1038.000000  3865.000000  2072.400329
re2_01    1004.000000  5518.000000  2166.834157
re3_01    1012.000000  6369.000000  2239.166392
red_01    1020.000000  4056.000000  1845.095552
swir1_01  1020.000000  4883.000000  2369.537068
swir2_01  1013.000000  4650.000000  2063.404723
VH_02      -47.843876   -10.918949   -25.704491
VV_02      -32.822712     1.086390   -15.515577
blue_02    975.000000  7251.000000  1675.332784
green_02  1071.000000  6959.000000  1908.510159
nir_02     992.000000  7882.000000  2341.576606
nira_02    925.000000  7437.000000  2278.833059
re1_02    1031.000000  699

#  Spectral indices per month (computed on raw wide data)


In [27]:
# ============================================================
#  Spectral indices per month (computed on raw wide data)
# ============================================================
NA_VAL = -9999

months = [f"{m:02d}" for m in range(1, 13)]
s1_bands = ['VH', 'VV']
s2_bands = ['blue','green','nir','nira','re1','re2','re3','red','swir1','swir2']

def add_indices(df):
    df = df.copy()
    for m in months:
        nir  = df[f'nir_{m}'].replace(NA_VAL, np.nan)
        red  = df[f'red_{m}'].replace(NA_VAL, np.nan)
        green= df[f'green_{m}'].replace(NA_VAL, np.nan)
        swir1= df[f'swir1_{m}'].replace(NA_VAL, np.nan)
        vh   = df[f'VH_{m}'].replace(NA_VAL, np.nan)
        vv   = df[f'VV_{m}'].replace(NA_VAL, np.nan)

        df[f'ndvi_{m}']  = (nir - red) / (nir + red + 1e-6)
        df[f'ndwi_{m}']  = (green - nir) / (green + nir + 1e-6)      # McFeeters
        df[f'mndwi_{m}'] = (green - swir1) / (green + swir1 + 1e-6) # better for turbid/pond water
        df[f'vhvv_{m}']  = vh - vv   # ratio in dB space = difference
    return df

train_idx = add_indices(train)
test_idx  = add_indices(test)

new_index_cols = [c for c in train_idx.columns if any(c.startswith(p) for p in ['ndvi_','ndwi_','mndwi_','vhvv_'])]
print("New index columns added:", len(new_index_cols))
print(new_index_cols[:8])

# sanity check: index values should be NaN exactly where source bands were -9999
print("\nTrain NaN count in ndvi_01 (should be 0):", train_idx['ndvi_01'].isna().sum())
print("Test NaN count in ndvi_01 (should be ~875/1000 rows, matches earlier missing %):", test_idx['ndvi_01'].isna().sum())

New index columns added: 48
['ndvi_01', 'ndwi_01', 'mndwi_01', 'vhvv_01', 'ndvi_02', 'ndwi_02', 'mndwi_02', 'vhvv_02']

Train NaN count in ndvi_01 (should be 0): 0
Test NaN count in ndvi_01 (should be ~875/1000 rows, matches earlier missing %): 902


#  Aggregation function: wide (per-month) -> per-row summary stats


In [28]:
# ============================================================
#  Aggregation function: wide (per-month) -> per-row summary stats
# Works identically on train (12 valid months) and test (4-6 valid months)
# ============================================================
all_bands = s1_bands + s2_bands + ['ndvi','ndwi','mndwi','vhvv']

def aggregate_features(df):
    out = pd.DataFrame(index=df.index)
    # replace -9999 with NaN for raw bands only (index cols already have NaN)
    work = df.copy()
    for b in s1_bands + s2_bands:
        cols = [f'{b}_{m}' for m in months]
        work[cols] = work[cols].replace(NA_VAL, np.nan)

    for b in all_bands:
        cols = [f'{b}_{m}' for m in months]
        vals = work[cols].values.astype(float)  # shape (n_rows, 12)

        out[f'{b}_mean']   = np.nanmean(vals, axis=1)
        out[f'{b}_std']    = np.nanstd(vals, axis=1)
        out[f'{b}_min']    = np.nanmin(vals, axis=1)
        out[f'{b}_max']    = np.nanmax(vals, axis=1)
        out[f'{b}_median'] = np.nanmedian(vals, axis=1)
        out[f'{b}_range']  = out[f'{b}_max'] - out[f'{b}_min']

        # first & last valid value in the row (captures trend endpoints)
        def first_valid(row):
            idx = np.where(~np.isnan(row))[0]
            return row[idx[0]] if len(idx) else np.nan
        def last_valid(row):
            idx = np.where(~np.isnan(row))[0]
            return row[idx[-1]] if len(idx) else np.nan

        out[f'{b}_first'] = np.apply_along_axis(first_valid, 1, vals)
        out[f'{b}_last']  = np.apply_along_axis(last_valid, 1, vals)
        out[f'{b}_trend'] = out[f'{b}_last'] - out[f'{b}_first']

    # meta features: how many months were active, for which bands
    s1_valid_count = np.sum(~np.isnan(work[[f'VH_{m}' for m in months]].values), axis=1)
    s2_valid_count = np.sum(~np.isnan(work[[f'blue_{m}' for m in months]].values), axis=1)
    out['n_active_s1_months'] = s1_valid_count
    out['n_active_s2_months'] = s2_valid_count
    out['s1_s2_month_gap']    = s1_valid_count - s2_valid_count

    return out

train_agg = aggregate_features(train_idx)
test_agg  = aggregate_features(test_idx)

print("Train agg shape:", train_agg.shape)
print("Test agg shape :", test_agg.shape)
print("\nAny inf values?", np.isinf(train_agg.values).sum(), np.isinf(test_agg.values).sum())
print("\nSample columns:", train_agg.columns.tolist()[:15])

/tmp/ipykernel_1318/2894850507.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f'{b}_std']    = np.nanstd(vals, axis=1)
/tmp/ipykernel_1318/2894850507.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f'{b}_min']    = np.nanmin(vals, axis=1)
/tmp/ipykernel_1318/2894850507.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmente

Train agg shape: (1821, 147)
Test agg shape : (1030, 147)

Any inf values? 0 0

Sample columns: ['VH_mean', 'VH_std', 'VH_min', 'VH_max', 'VH_median', 'VH_range', 'VH_first', 'VH_last', 'VH_trend', 'VV_mean', 'VV_std', 'VV_min', 'VV_max', 'VV_median', 'VV_range']


/tmp/ipykernel_1318/2894850507.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f'{b}_std']    = np.nanstd(vals, axis=1)
/tmp/ipykernel_1318/2894850507.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f'{b}_min']    = np.nanmin(vals, axis=1)
/tmp/ipykernel_1318/2894850507.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmente

# Quick baseline: aggregated features, NO augmentation yet


In [29]:
# ============================================================
# Quick baseline: aggregated features, NO augmentation yet
# Just to get a first honest CV number before we add the masking-simulation step
# ============================================================
!pip install lightgbm -q
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

X = train_agg.replace([np.inf, -np.inf], np.nan).fillna(-999)
y = train['label'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_pred = np.zeros(len(X))
oof_proba = np.zeros(len(X))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(50, verbose=False)])

    proba = model.predict_proba(X_va)[:, 1]
    oof_proba[va_idx] = proba
    oof_pred[va_idx] = (proba >= 0.5).astype(int)

f1 = f1_score(y, oof_pred)
auc = roc_auc_score(y, oof_proba)
blended = 0.6*f1 + 0.4*auc
print(f"OOF F1:  {f1:.4f}")
print(f"OOF AUC: {auc:.4f}")
print(f"Blended (leaderboard-style) score: {blended:.4f}")

OOF F1:  0.9740
OOF AUC: 0.9951
Blended (leaderboard-style) score: 0.9824


#  Rewritten aggregate_features (dict-based, no fragmentation)


In [30]:
# ============================================================
#  Rewritten aggregate_features (dict-based, no fragmentation)
# ============================================================
def aggregate_features(df):
    work = df.copy()
    for b in s1_bands + s2_bands:
        cols = [f'{b}_{m}' for m in months]
        work[cols] = work[cols].replace(NA_VAL, np.nan)

    out = {}
    for b in all_bands:
        cols = [f'{b}_{m}' for m in months]
        vals = work[cols].values.astype(float)

        out[f'{b}_mean']   = np.nanmean(vals, axis=1)
        out[f'{b}_std']    = np.nanstd(vals, axis=1)
        out[f'{b}_min']    = np.nanmin(vals, axis=1)
        out[f'{b}_max']    = np.nanmax(vals, axis=1)
        out[f'{b}_median'] = np.nanmedian(vals, axis=1)
        out[f'{b}_range']  = out[f'{b}_max'] - out[f'{b}_min']

        valid_mask = ~np.isnan(vals)
        first_idx = valid_mask.argmax(axis=1)  # first True index (0 if none valid)
        # last valid index: reverse the mask
        last_idx = vals.shape[1] - 1 - valid_mask[:, ::-1].argmax(axis=1)
        has_valid = valid_mask.any(axis=1)

        rows = np.arange(vals.shape[0])
        first_vals = np.where(has_valid, vals[rows, first_idx], np.nan)
        last_vals  = np.where(has_valid, vals[rows, last_idx], np.nan)

        out[f'{b}_first'] = first_vals
        out[f'{b}_last']  = last_vals
        out[f'{b}_trend'] = last_vals - first_vals

    s1_valid_count = np.sum(~np.isnan(work[[f'VH_{m}' for m in months]].values), axis=1)
    s2_valid_count = np.sum(~np.isnan(work[[f'blue_{m}' for m in months]].values), axis=1)
    out['n_active_s1_months'] = s1_valid_count
    out['n_active_s2_months'] = s2_valid_count
    out['s1_s2_month_gap']    = s1_valid_count - s2_valid_count

    return pd.DataFrame(out, index=df.index)

# rebuild test_agg with the fixed function (should be identical values, just faster)
test_agg = aggregate_features(test_idx)
print("test_agg rebuilt:", test_agg.shape)

test_agg rebuilt: (1030, 147)


#Simulate test-like masking on TRAIN (the key fix)


In [31]:
# ============================================================
#Simulate test-like masking on TRAIN (the key fix)
# ============================================================
NA_VAL = -9999
raw_bands = s1_bands + s2_bands  # 12 raw bands, NOT indices (indices recomputed after masking)

# Empirically observed rate of "S1 present, S2 fully missing" within an active month
S2_DROPOUT_RATE = 320 / (345*4 + 343*5 + 342*6)  # ≈ 0.0622
print("S2 dropout rate used:", S2_DROPOUT_RATE)

# Empirical start-month weights from TEST (index 0 = Jan)
start_weights_full = {0:129, 1:131, 2:130, 3:129, 4:130, 5:130, 6:130, 7:82, 8:39}
length_choices = np.array([4, 5, 6])

def build_masked_train(train_df, n_aug=15, seed=42):
    rng = np.random.default_rng(seed)
    n = len(train_df)
    total = n * n_aug

    rep_idx = np.repeat(np.arange(n), n_aug)
    aug = train_df.iloc[rep_idx].reset_index(drop=True).copy()
    aug['group_id'] = train_df['ID'].values[rep_idx]  # keep original row grouped

    lengths = rng.choice(length_choices, size=total, replace=True)

    starts = np.zeros(total, dtype=int)
    for L in length_choices:
        mask_L = (lengths == L)
        max_start = 12 - L
        valid_starts = np.array([s for s in start_weights_full if s <= max_start])
        w = np.array([start_weights_full[s] for s in valid_starts], dtype=float)
        w /= w.sum()
        starts[mask_L] = rng.choice(valid_starts, size=mask_L.sum(), p=w)

    month_idx = np.arange(12)
    active_mask = (month_idx[None, :] >= starts[:, None]) & (month_idx[None, :] < (starts + lengths)[:, None])

    s2_drop = (rng.random((total, 12)) < S2_DROPOUT_RATE) & active_mask

    for mi, m in enumerate(months):
        inactive_rows = ~active_mask[:, mi]
        s2_drop_rows = s2_drop[:, mi]
        for b in raw_bands:
            col = f'{b}_{m}'
            vals = aug[col].values.copy()
            vals[inactive_rows] = NA_VAL
            if b in s2_bands:
                vals[s2_drop_rows] = NA_VAL
            aug[col] = vals

    aug['sim_window_len'] = lengths
    aug['sim_window_start'] = starts
    return aug

train_aug_wide = build_masked_train(train, n_aug=15, seed=42)
print("Augmented train wide shape:", train_aug_wide.shape)
print(train_aug_wide['sim_window_len'].value_counts())

S2 dropout rate used: 0.062172139110161256
Augmented train wide shape: (27315, 149)
sim_window_len
4    9212
5    9112
6    8991
Name: count, dtype: int64


/tmp/ipykernel_1318/3174094469.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aug['sim_window_len'] = lengths
/tmp/ipykernel_1318/3174094469.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aug['sim_window_start'] = starts


#  — Recompute indices + aggregates on the masked augmented train


In [32]:
# ============================================================
#  — Recompute indices + aggregates on the masked augmented train
# ============================================================
train_aug_idx = add_indices(train_aug_wide)
train_aug_agg = aggregate_features(train_aug_idx)
train_aug_agg['label']    = train_aug_wide['label'].values
train_aug_agg['group_id'] = train_aug_wide['group_id'].values

print("train_aug_agg shape:", train_aug_agg.shape)

# --- Sanity check: does augmented-train distribution now resemble TEST? ---
compare_cols = ['n_active_s1_months', 'n_active_s2_months', 'VH_mean', 'ndvi_mean', 'mndwi_mean']
print("\n--- TEST feature stats ---")
print(test_agg[compare_cols].describe().T[['mean','std','min','max']])
print("\n--- AUGMENTED TRAIN feature stats (should now look similar) ---")
print(train_aug_agg[compare_cols].describe().T[['mean','std','min','max']])

train_aug_agg shape: (27315, 149)

--- TEST feature stats ---
                         mean       std        min        max
n_active_s1_months   4.997087  0.817086   4.000000   6.000000
n_active_s2_months   4.686408  0.967755   2.000000   6.000000
VH_mean            -25.760882  5.780997 -35.305646 -10.228434
ndvi_mean            0.048735  0.133866  -0.151000   0.530039
mndwi_mean          -0.033749  0.194120  -0.436375   0.356618

--- AUGMENTED TRAIN feature stats (should now look similar) ---
                         mean       std        min        max
n_active_s1_months   4.991909  0.816314   4.000000   6.000000
n_active_s2_months   4.686290  0.934339   1.000000   6.000000
VH_mean            -25.658264  5.933236 -42.511397 -10.215895
ndvi_mean            0.100390  0.179014  -0.229136   0.667060
mndwi_mean          -0.028311  0.211035  -0.404943   0.361432


#  Retrain with GroupKFold on augmented data (honest CV)


In [33]:
# ============================================================
#  Retrain with GroupKFold on augmented data (honest CV)
# ============================================================
from sklearn.model_selection import GroupKFold

feature_cols_agg = [c for c in train_aug_agg.columns if c not in ['label', 'group_id']]
X = train_aug_agg[feature_cols_agg].replace([np.inf, -np.inf], np.nan).fillna(-999)
y = train_aug_agg['label'].values
groups = train_aug_agg['group_id'].values

gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(X))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_proba[va_idx] = model.predict_proba(X_va)[:, 1]

oof_pred = (oof_proba >= 0.5).astype(int)
f1 = f1_score(y, oof_pred)
auc = roc_auc_score(y, oof_proba)
print(f"OOF F1:  {f1:.4f}")
print(f"OOF AUC: {auc:.4f}")
print(f"Blended: {0.6*f1 + 0.4*auc:.4f}")

OOF F1:  0.9663
OOF AUC: 0.9937
Blended: 0.9772


# Adversarial validation: how different are train vs test really?


In [34]:
# ============================================================
# Adversarial validation: how different are train vs test really?
# ============================================================
from sklearn.model_selection import StratifiedKFold

# Use masked-augmented train (matches test's missingness pattern) vs real test,
# so any signal the model finds is due to VALUE shift, not missingness shift.
adv_train = train_aug_agg[feature_cols_agg].replace([np.inf,-np.inf], np.nan).fillna(-999).copy()
adv_test  = test_agg[feature_cols_agg].replace([np.inf,-np.inf], np.nan).fillna(-999).copy()

adv_X = pd.concat([adv_train, adv_test], axis=0).reset_index(drop=True)
adv_y = np.array([0]*len(adv_train) + [1]*len(adv_test))  # 1 = test

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
adv_oof = np.zeros(len(adv_X))
importances = np.zeros(len(feature_cols_agg))

for tr_idx, va_idx in skf.split(adv_X, adv_y):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                            random_state=42, verbose=-1)
    m.fit(adv_X.iloc[tr_idx], adv_y[tr_idx])
    adv_oof[va_idx] = m.predict_proba(adv_X.iloc[va_idx])[:, 1]
    importances += m.feature_importances_

adv_auc = roc_auc_score(adv_y, adv_oof)
print(f"Adversarial AUC (0.5 = indistinguishable, 1.0 = fully separable): {adv_auc:.4f}")

imp_df = pd.Series(importances, index=feature_cols_agg).sort_values(ascending=False)
print("\nTop 25 features driving train/test separability (i.e. most shifted):")
print(imp_df.head(25))

Adversarial AUC (0.5 = indistinguishable, 1.0 = fully separable): 0.9923

Top 25 features driving train/test separability (i.e. most shifted):
mndwi_min       852.0
swir1_min       787.0
VV_median       751.0
vhvv_max        722.0
vhvv_median     693.0
mndwi_last      676.0
blue_mean       667.0
blue_min        652.0
swir1_last      648.0
ndwi_max        619.0
vhvv_mean       619.0
mndwi_median    615.0
swir2_std       577.0
VH_min          572.0
ndvi_min        562.0
VV_min          557.0
ndvi_median     551.0
blue_max        546.0
nir_min         545.0
mndwi_range     530.0
ndvi_max        529.0
VH_range        528.0
red_min         517.0
VV_last         512.0
ndwi_min        511.0
dtype: float64


#  Compare raw mean shift for the top drifting features


In [35]:
# ============================================================
#  Compare raw mean shift for the top drifting features
# ============================================================
top_shift_feats = imp_df.head(20).index.tolist()
comparison = pd.DataFrame({
    'train_aug_mean': train_aug_agg[top_shift_feats].mean(),
    'test_mean':      test_agg[top_shift_feats].mean(),
})
comparison['abs_diff'] = (comparison['train_aug_mean'] - comparison['test_mean']).abs()
comparison['pct_diff'] = comparison['abs_diff'] / (comparison['train_aug_mean'].abs() + 1e-6) * 100
print(comparison.sort_values('pct_diff', ascending=False))

              train_aug_mean    test_mean    abs_diff     pct_diff
ndwi_max            0.002025     0.036874    0.034850  1720.522828
ndvi_min            0.010645    -0.008087    0.018733   175.954611
ndvi_median         0.095686     0.046092    0.049594    51.829288
swir2_std         282.097788   383.496308  101.398520    35.944457
mndwi_last         -0.037570    -0.045136    0.007566    20.138077
mndwi_min          -0.094894    -0.113662    0.018768    19.777591
mndwi_range         0.127236     0.152219    0.024983    19.634977
mndwi_median       -0.025318    -0.029475    0.004157    16.417184
vhvv_max           -6.615848    -5.537335    1.078513    16.301962
vhvv_median        -9.251803    -7.772378    1.479425    15.990665
vhvv_mean          -9.357471    -7.893694    1.463777    15.642869
nir_min          2045.281677  1786.743689  258.537987    12.640703
VV_median         -16.436919   -18.127823    1.690905    10.287237
blue_max         1950.721106  2132.606796  181.885690     9.32

#  Per-domain quantile normalization (marginal distribution alignment)


In [36]:
# ============================================================
#  Per-domain quantile normalization (marginal distribution alignment)
# ============================================================
from sklearn.preprocessing import QuantileTransformer

def quantile_align(train_df, test_df, cols):
    train_out = train_df.copy()
    test_out  = test_df.copy()
    for c in cols:
        tr_vals = train_df[c].values.astype(float)
        te_vals = test_df[c].values.astype(float)

        tr_mask = ~np.isnan(tr_vals)
        te_mask = ~np.isnan(te_vals)

        if tr_mask.sum() < 10 or te_mask.sum() < 10:
            continue  # not enough data to fit a meaningful transform

        qt_tr = QuantileTransformer(output_distribution='normal',
                                     n_quantiles=min(1000, tr_mask.sum()),
                                     random_state=42)
        qt_te = QuantileTransformer(output_distribution='normal',
                                     n_quantiles=min(1000, te_mask.sum()),
                                     random_state=42)

        tr_transformed = tr_vals.copy()
        te_transformed = te_vals.copy()

        tr_transformed[tr_mask] = qt_tr.fit_transform(tr_vals[tr_mask].reshape(-1,1)).ravel()
        te_transformed[te_mask] = qt_te.fit_transform(te_vals[te_mask].reshape(-1,1)).ravel()

        train_out[c] = tr_transformed
        test_out[c]  = te_transformed
    return train_out, test_out

# Don't fill NaN with -999 anymore — keep NaN, let LightGBM handle it natively
train_aug_agg_raw = train_aug_agg.copy()  # has label, group_id + features, NaNs where missing
test_agg_raw = test_agg.copy()

train_qn, test_qn = quantile_align(train_aug_agg_raw, test_agg_raw, feature_cols_agg)
print("Quantile-normalized shapes:", train_qn.shape, test_qn.shape)

Quantile-normalized shapes: (27315, 149) (1030, 147)


#  Re-run adversarial validation on quantile-normalized features


In [37]:
# ============================================================
#  Re-run adversarial validation on quantile-normalized features
# ============================================================
adv_X2 = pd.concat([train_qn[feature_cols_agg], test_qn[feature_cols_agg]], axis=0).reset_index(drop=True)
adv_y2 = np.array([0]*len(train_qn) + [1]*len(test_qn))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
adv_oof2 = np.zeros(len(adv_X2))

for tr_idx, va_idx in skf.split(adv_X2, adv_y2):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                            random_state=42, verbose=-1)
    m.fit(adv_X2.iloc[tr_idx], adv_y2[tr_idx])
    adv_oof2[va_idx] = m.predict_proba(adv_X2.iloc[va_idx])[:, 1]

adv_auc2 = roc_auc_score(adv_y2, adv_oof2)
print(f"Adversarial AUC AFTER quantile alignment: {adv_auc2:.4f}  (was 0.9923)")

Adversarial AUC AFTER quantile alignment: 0.9993  (was 0.9923)


#  Retrain pond classifier on quantile-normalized features, GroupKFold


In [38]:
# ============================================================
#  Retrain pond classifier on quantile-normalized features, GroupKFold
# ============================================================
X2 = train_qn[feature_cols_agg]  # NaN kept as-is, LightGBM handles natively
y2 = train_qn['label'].values
groups2 = train_qn['group_id'].values

gkf = GroupKFold(n_splits=5)
oof_proba2 = np.zeros(len(X2))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X2, y2, groups2)):
    X_tr, X_va = X2.iloc[tr_idx], X2.iloc[va_idx]
    y_tr, y_va = y2[tr_idx], y2[va_idx]

    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_proba2[va_idx] = model.predict_proba(X_va)[:, 1]

oof_pred2 = (oof_proba2 >= 0.5).astype(int)
f1_2 = f1_score(y2, oof_pred2)
auc_2 = roc_auc_score(y2, oof_proba2)
print(f"OOF F1:  {f1_2:.4f}")
print(f"OOF AUC: {auc_2:.4f}")
print(f"Blended: {0.6*f1_2 + 0.4*auc_2:.4f}")

OOF F1:  0.9656
OOF AUC: 0.9937
Blended: 0.9769


#  Fix: fit QuantileTransformer on deduplicated data, then transform everything


In [39]:
# ============================================================
#  Fix: fit QuantileTransformer on deduplicated data, then transform everything
# ============================================================
from sklearn.preprocessing import QuantileTransformer

# Step 1: build ONE masked draw per original train row (1821 rows, no duplication)
train_fit_wide = build_masked_train(train, n_aug=1, seed=123)
train_fit_idx  = add_indices(train_fit_wide)
train_fit_agg  = aggregate_features(train_fit_idx)   # 1821 rows, distinct locations, test-like masking

print("Dedup fit-sample shape:", train_fit_agg.shape)

def fit_quantile_mappers(fit_df, cols):
    mappers = {}
    for c in cols:
        vals = fit_df[c].values.astype(float)
        mask = ~np.isnan(vals)
        if mask.sum() < 10:
            mappers[c] = None
            continue
        qt = QuantileTransformer(output_distribution='normal',
                                  n_quantiles=min(1000, mask.sum()),
                                  random_state=42)
        qt.fit(vals[mask].reshape(-1, 1))
        mappers[c] = qt
    return mappers

def apply_quantile_mappers(df, cols, mappers):
    out = df.copy()
    for c in cols:
        qt = mappers[c]
        if qt is None:
            continue
        vals = df[c].values.astype(float)
        mask = ~np.isnan(vals)
        transformed = vals.copy()
        transformed[mask] = qt.transform(vals[mask].reshape(-1, 1)).ravel()
        out[c] = transformed
    return out

# Fit separately per domain, on DEDUPLICATED representative samples
train_mappers = fit_quantile_mappers(train_fit_agg, feature_cols_agg)
test_mappers  = fit_quantile_mappers(test_agg, feature_cols_agg)

# Now just TRANSFORM (no refit) the full augmented train and the real test
train_qn2 = apply_quantile_mappers(train_aug_agg, feature_cols_agg, train_mappers)
test_qn2  = apply_quantile_mappers(test_agg, feature_cols_agg, test_mappers)

print("Transformed shapes:", train_qn2.shape, test_qn2.shape)

/tmp/ipykernel_1318/3174094469.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aug['sim_window_len'] = lengths
/tmp/ipykernel_1318/3174094469.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aug['sim_window_start'] = starts


Dedup fit-sample shape: (1821, 147)
Transformed shapes: (27315, 149) (1030, 147)


#  Re-run adversarial validation (should now actually drop)


In [40]:
# ============================================================
#  Re-run adversarial validation (should now actually drop)
# ============================================================
adv_X3 = pd.concat([train_qn2[feature_cols_agg], test_qn2[feature_cols_agg]], axis=0).reset_index(drop=True)
adv_y3 = np.array([0]*len(train_qn2) + [1]*len(test_qn2))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
adv_oof3 = np.zeros(len(adv_X3))

for tr_idx, va_idx in skf.split(adv_X3, adv_y3):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                            random_state=42, verbose=-1)
    m.fit(adv_X3.iloc[tr_idx], adv_y3[tr_idx])
    adv_oof3[va_idx] = m.predict_proba(adv_X3.iloc[va_idx])[:, 1]

adv_auc3 = roc_auc_score(adv_y3, adv_oof3)
print(f"Adversarial AUC after FIXED quantile alignment: {adv_auc3:.4f}")
print(f"(sequence: 0.9923 raw -> 0.9993 buggy-transform -> {adv_auc3:.4f} fixed)")

Adversarial AUC after FIXED quantile alignment: 0.9994
(sequence: 0.9923 raw -> 0.9993 buggy-transform -> 0.9994 fixed)


#  — Retrain pond classifier on properly-aligned features


In [41]:
# ============================================================
#  — Retrain pond classifier on properly-aligned features
# ============================================================
X3 = train_qn2[feature_cols_agg]
y3 = train_qn2['label'].values
groups3 = train_qn2['group_id'].values

gkf = GroupKFold(n_splits=5)
oof_proba3 = np.zeros(len(X3))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X3, y3, groups3)):
    X_tr, X_va = X3.iloc[tr_idx], X3.iloc[va_idx]
    y_tr, y_va = y3[tr_idx], y3[va_idx]

    model = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=42, verbose=-1
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_proba3[va_idx] = model.predict_proba(X_va)[:, 1]

oof_pred3 = (oof_proba3 >= 0.5).astype(int)
f1_3 = f1_score(y3, oof_pred3)
auc_3 = roc_auc_score(y3, oof_proba3)
print(f"OOF F1:  {f1_3:.4f}")
print(f"OOF AUC: {auc_3:.4f}")
print(f"Blended: {0.6*f1_3 + 0.4*auc_3:.4f}")

OOF F1:  0.9657
OOF AUC: 0.9937
Blended: 0.9769


# Add brightness-normalized (calibration-invariant) band features


In [42]:
# ============================================================
# Add brightness-normalized (calibration-invariant) band features
# ============================================================
def add_normalized_bands(df):
    df = df.copy()
    for m in months:
        b   = df[f'blue_{m}'].replace(NA_VAL, np.nan)
        g   = df[f'green_{m}'].replace(NA_VAL, np.nan)
        r   = df[f'red_{m}'].replace(NA_VAL, np.nan)
        nir = df[f'nir_{m}'].replace(NA_VAL, np.nan)
        s1  = df[f'swir1_{m}'].replace(NA_VAL, np.nan)
        s2  = df[f'swir2_{m}'].replace(NA_VAL, np.nan)
        vh  = df[f'VH_{m}'].replace(NA_VAL, np.nan)
        vv  = df[f'VV_{m}'].replace(NA_VAL, np.nan)

        brightness = b + g + r + nir + s1 + s2 + 1e-6
        df[f'blue_norm_{m}']  = b   / brightness
        df[f'green_norm_{m}'] = g   / brightness
        df[f'red_norm_{m}']   = r   / brightness
        df[f'nir_norm_{m}']   = nir / brightness
        df[f'swir1_norm_{m}'] = s1  / brightness
        df[f'swir2_norm_{m}'] = s2  / brightness

        # SAR: ratio instead of raw dB difference (already have vhvv=diff; add ratio in linear domain)
        df[f'vh_frac_{m}'] = vh / (vh + vv - 1e-6)  # relative share, calibration-drift-resistant
    return df

train_fit_norm = add_normalized_bands(add_indices(train_fit_wide))   # dedup, test-like mask, for fitting reference
train_aug_norm = add_normalized_bands(train_aug_idx)
test_norm      = add_normalized_bands(test_idx)

norm_bands = ['blue_norm','green_norm','red_norm','nir_norm','swir1_norm','swir2_norm','vh_frac']

def aggregate_norm(df):
    out = {}
    for b in norm_bands:
        cols = [f'{b}_{m}' for m in months]
        vals = df[cols].values.astype(float)
        out[f'{b}_mean']   = np.nanmean(vals, axis=1)
        out[f'{b}_std']    = np.nanstd(vals, axis=1)
        out[f'{b}_min']    = np.nanmin(vals, axis=1)
        out[f'{b}_max']    = np.nanmax(vals, axis=1)
        out[f'{b}_median'] = np.nanmedian(vals, axis=1)
    return pd.DataFrame(out, index=df.index)

train_norm_agg = aggregate_norm(train_aug_norm)
test_norm_agg  = aggregate_norm(test_norm)

print("Normalized feature block shapes:", train_norm_agg.shape, test_norm_agg.shape)

Normalized feature block shapes: (27315, 35) (1030, 35)


#  Adversarial validation on normalized features ALONE


In [43]:
# ============================================================
#  Adversarial validation on normalized features ALONE
# (tells us: does calibration-invariant framing actually reduce separability?)
# ============================================================
norm_cols = train_norm_agg.columns.tolist()
adv_X4 = pd.concat([train_norm_agg, test_norm_agg], axis=0).reset_index(drop=True)
adv_y4 = np.array([0]*len(train_norm_agg) + [1]*len(test_norm_agg))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
adv_oof4 = np.zeros(len(adv_X4))
for tr_idx, va_idx in skf.split(adv_X4, adv_y4):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                            random_state=42, verbose=-1)
    m.fit(adv_X4.iloc[tr_idx], adv_y4[tr_idx])
    adv_oof4[va_idx] = m.predict_proba(adv_X4.iloc[va_idx])[:, 1]

adv_auc4 = roc_auc_score(adv_y4, adv_oof4)
print(f"Adversarial AUC on NORMALIZED-only features: {adv_auc4:.4f}  (raw-feature version was 0.9923-0.9994)")

Adversarial AUC on NORMALIZED-only features: 0.9799  (raw-feature version was 0.9923-0.9994)


#  Combine original + normalized features, retrain pond classifier


In [44]:
# ============================================================
#  Combine original + normalized features, retrain pond classifier
# ============================================================
train_combined = pd.concat([train_aug_agg.reset_index(drop=True), train_norm_agg.reset_index(drop=True)], axis=1)
test_combined  = pd.concat([test_agg.reset_index(drop=True),      test_norm_agg.reset_index(drop=True)], axis=1)

feature_cols_v2 = [c for c in train_combined.columns if c not in ['label', 'group_id']]
X4 = train_combined[feature_cols_v2]
y4 = train_combined['label'].values
groups4 = train_combined['group_id'].values

gkf = GroupKFold(n_splits=5)
oof_proba4 = np.zeros(len(X4))
for tr_idx, va_idx in gkf.split(X4, y4, groups4):
    model = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=42, verbose=-1
    )
    model.fit(X4.iloc[tr_idx], y4[tr_idx], eval_set=[(X4.iloc[va_idx], y4[va_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_proba4[va_idx] = model.predict_proba(X4.iloc[va_idx])[:, 1]

oof_pred4 = (oof_proba4 >= 0.5).astype(int)
f1_4 = f1_score(y4, oof_pred4)
auc_4 = roc_auc_score(y4, oof_proba4)
print(f"OOF F1:  {f1_4:.4f}")
print(f"OOF AUC: {auc_4:.4f}")
print(f"Blended: {0.6*f1_4 + 0.4*auc_4:.4f}")

OOF F1:  0.9688
OOF AUC: 0.9932
Blended: 0.9785


#  Adversarial reweighting + drop worst-shifted raw features


In [45]:
# ============================================================
#  Adversarial reweighting + drop worst-shifted raw features
# ============================================================
# Refit one clean adversarial model on the (now larger) combined feature set
# to get per-row weights for the REAL pond classifier
adv_model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                random_state=42, verbose=-1)
adv_model.fit(pd.concat([train_combined[feature_cols_v2], test_combined[feature_cols_v2]], axis=0),
              np.array([0]*len(train_combined) + [1]*len(test_combined)))

# Predict p(test) for TRAIN rows only -> importance weight
train_p_test = adv_model.predict_proba(train_combined[feature_cols_v2])[:, 1]
train_p_test = np.clip(train_p_test, 0.02, 0.98)  # avoid extreme weights
sample_weights = train_p_test / (1 - train_p_test)
sample_weights = sample_weights / sample_weights.mean()  # normalize around 1.0

print("Sample weight stats:", pd.Series(sample_weights).describe())

# Worst offenders from earlier (highest pct_diff / most artificial-looking)
drop_feats = ['ndwi_max', 'ndvi_min', 'swir2_std', 'nir_min', 'blue_max']
feature_cols_v3 = [c for c in feature_cols_v2 if c not in drop_feats]
print(f"\nDropped {len(drop_feats)} worst-shifted features, {len(feature_cols_v3)} remain")

Sample weight stats: count    27315.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
dtype: float64

Dropped 5 worst-shifted features, 177 remain


#  Retrain with sample weights + pruned features, GroupKFold


In [46]:
# ============================================================
#  Retrain with sample weights + pruned features, GroupKFold
# ============================================================
X5 = train_combined[feature_cols_v3]
y5 = train_combined['label'].values
groups5 = train_combined['group_id'].values
w5 = sample_weights

gkf = GroupKFold(n_splits=5)
oof_proba5 = np.zeros(len(X5))

for tr_idx, va_idx in gkf.split(X5, y5, groups5):
    model = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=42, verbose=-1
    )
    model.fit(X5.iloc[tr_idx], y5[tr_idx], sample_weight=w5[tr_idx],
              eval_set=[(X5.iloc[va_idx], y5[va_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_proba5[va_idx] = model.predict_proba(X5.iloc[va_idx])[:, 1]

oof_pred5 = (oof_proba5 >= 0.5).astype(int)
f1_5 = f1_score(y5, oof_pred5)
auc_5 = roc_auc_score(y5, oof_proba5)
print(f"OOF F1:  {f1_5:.4f}")
print(f"OOF AUC: {auc_5:.4f}")
print(f"Blended: {0.6*f1_5 + 0.4*auc_5:.4f}")

OOF F1:  0.9680
OOF AUC: 0.9936
Blended: 0.9782


# FIXED: out-of-fold adversarial weights (no self-prediction leakage)


In [47]:
# ============================================================
# FIXED: out-of-fold adversarial weights (no self-prediction leakage)
# ============================================================
adv_X_full = pd.concat([train_combined[feature_cols_v3], test_combined[feature_cols_v3]], axis=0).reset_index(drop=True)
adv_y_full = np.array([0]*len(train_combined) + [1]*len(test_combined))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
adv_oof_full = np.zeros(len(adv_X_full))

for tr_idx, va_idx in skf.split(adv_X_full, adv_y_full):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                            random_state=42, verbose=-1)
    m.fit(adv_X_full.iloc[tr_idx], adv_y_full[tr_idx])
    adv_oof_full[va_idx] = m.predict_proba(adv_X_full.iloc[va_idx])[:, 1]

adv_auc_full = roc_auc_score(adv_y_full, adv_oof_full)
print(f"Adversarial AUC (177 combined features, OOF): {adv_auc_full:.4f}")

# Extract OOF p(test) for TRAIN rows only (first len(train_combined) entries)
train_p_test_oof = adv_oof_full[:len(train_combined)]
train_p_test_oof = np.clip(train_p_test_oof, 0.02, 0.98)
sample_weights_v2 = train_p_test_oof / (1 - train_p_test_oof)
sample_weights_v2 = sample_weights_v2 / sample_weights_v2.mean()

print("\nFixed sample weight stats:")
print(pd.Series(sample_weights_v2).describe())
print("\nWeight distribution by original window length (do longer/shorter windows look more test-like?):")
print(pd.Series(sample_weights_v2).groupby(train_aug_wide['sim_window_len'].values).describe())

Adversarial AUC (177 combined features, OOF): 0.9933

Fixed sample weight stats:
count    27315.000000
mean         1.000000
std          1.394532
min          0.961928
25%          0.961928
50%          0.961928
75%          0.961928
max        112.002828
dtype: float64

Weight distribution by original window length (do longer/shorter windows look more test-like?):
    count      mean       std       min       25%       50%       75%  \
4  9212.0  1.023695  1.734517  0.961928  0.961928  0.961928  0.961928   
5  9112.0  0.972922  0.663986  0.961928  0.961928  0.961928  0.961928   
6  8991.0  1.003166  1.542052  0.961928  0.961928  0.961928  0.961928   

          max  
4  112.002828  
5   63.531881  
6   84.400496  


#  Retrain with FIXED out-of-fold weights, GroupKFold


In [48]:
# ============================================================
#  Retrain with FIXED out-of-fold weights, GroupKFold
# ============================================================
X6 = train_combined[feature_cols_v3]
y6 = train_combined['label'].values
groups6 = train_combined['group_id'].values
w6 = sample_weights_v2

gkf = GroupKFold(n_splits=5)
oof_proba6 = np.zeros(len(X6))

for tr_idx, va_idx in gkf.split(X6, y6, groups6):
    model = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=42, verbose=-1
    )
    model.fit(X6.iloc[tr_idx], y6[tr_idx], sample_weight=w6[tr_idx],
              eval_set=[(X6.iloc[va_idx], y6[va_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_proba6[va_idx] = model.predict_proba(X6.iloc[va_idx])[:, 1]

oof_pred6 = (oof_proba6 >= 0.5).astype(int)
f1_6 = f1_score(y6, oof_pred6)
auc_6 = roc_auc_score(y6, oof_proba6)
print(f"OOF F1:  {f1_6:.4f}")
print(f"OOF AUC: {auc_6:.4f}")
print(f"Blended: {0.6*f1_6 + 0.4*auc_6:.4f}")

OOF F1:  0.9675
OOF AUC: 0.9935
Blended: 0.9779


# Generate leaderboard submission using current best pipeline


In [49]:
# ============================================================
# Generate leaderboard submission using current best pipeline
# ============================================================
final_model = lgb.LGBMClassifier(
    n_estimators=int(model.best_iteration_ * 1.1) if hasattr(model, 'best_iteration_') else 1500,
    learning_rate=0.03, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced', random_state=42, verbose=-1
)
final_model.fit(X6, y6, sample_weight=w6)

test_proba = final_model.predict_proba(test_combined[feature_cols_v3])[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

submission = pd.DataFrame({
    'ID': test['ID'].values,
    'TargetF1': test_pred,
    'TargetRAUC': test_proba
})
print(submission.head())
print("\nPredicted positive rate:", submission['TargetF1'].mean())

submission.to_csv('/content/drive/MyDrive/submission_v1.csv', index=False)
print("Saved.")

                   ID  TargetF1  TargetRAUC
0  ID_TS_NEW_SBZAYD5I         1    0.975996
1  ID_TS_NEW_7SPRN3PB         1    0.995256
2  ID_TS_NEW_LZOWPHLC         1    0.908159
3  ID_TS_NEW_DN6TUF64         1    0.786429
4  ID_TS_NEW_95N82M49         0    0.016929

Predicted positive rate: 0.587378640776699
Saved.


#  Diagnose: what is the current model actually using?


In [50]:
# ============================================================
#  Diagnose: what is the current model actually using?
# ============================================================
final_model.fit(X6, y6, sample_weight=w6)  # refit on all data for inspection
imp = pd.Series(final_model.feature_importances_, index=feature_cols_v3).sort_values(ascending=False)
print("Top 30 features by importance in the CURRENT (overfit) model:")
print(imp.head(30))

print("\nHow many features have near-zero importance (candidates to drop)?")
print((imp < imp.max()*0.01).sum(), "out of", len(imp))

Top 30 features by importance in the CURRENT (overfit) model:
green_norm_max       459
red_norm_max         239
swir2_max            231
VV_min               223
VV_mean              182
swir2_norm_min       176
green_norm_median    157
VH_min               134
VV_max               132
nir_norm_max         132
swir2_min            115
green_max            110
re1_max              106
red_min              104
blue_min              99
blue_norm_min         97
VH_mean               92
green_min             91
re3_max               90
nir_norm_std          88
swir2_norm_median     87
swir2_median          84
vh_frac_min           78
vhvv_max              76
swir1_norm_min        75
green_norm_std        75
vh_frac_max           74
green_norm_mean       73
VV_median             71
VH_max                71
dtype: int32

How many features have near-zero importance (candidates to drop)?
9 out of 177


#  Curated, low-dimensional, robust feature set


In [51]:
# ============================================================
#  Curated, low-dimensional, robust feature set
# Prioritize: spectral indices (calibration-resistant) + normalized bands
# over raw reflectance / raw SAR values, and drop first/last/trend
# (noisy — depend heavily on exact window position)
# ============================================================
core_bands = ['ndvi', 'ndwi', 'mndwi', 'vhvv']  # indices, not raw reflectance
core_stats = ['mean', 'std', 'min', 'max', 'median']

curated_cols = []
for b in core_bands:
    for s in core_stats:
        col = f'{b}_{s}'
        if col in train_combined.columns:
            curated_cols.append(col)

# add the normalized-band block (already calibration-resistant) fully
curated_cols += [c for c in norm_cols if c in train_combined.columns]

# add meta features (window length/position signal — legitimately differs but useful)
curated_cols += ['n_active_s1_months', 'n_active_s2_months', 's1_s2_month_gap']

curated_cols = sorted(set(curated_cols))
print(f"Curated feature count: {len(curated_cols)} (down from {len(feature_cols_v3)})")
print(curated_cols)

Curated feature count: 58 (down from 177)
['blue_norm_max', 'blue_norm_mean', 'blue_norm_median', 'blue_norm_min', 'blue_norm_std', 'green_norm_max', 'green_norm_mean', 'green_norm_median', 'green_norm_min', 'green_norm_std', 'mndwi_max', 'mndwi_mean', 'mndwi_median', 'mndwi_min', 'mndwi_std', 'n_active_s1_months', 'n_active_s2_months', 'ndvi_max', 'ndvi_mean', 'ndvi_median', 'ndvi_min', 'ndvi_std', 'ndwi_max', 'ndwi_mean', 'ndwi_median', 'ndwi_min', 'ndwi_std', 'nir_norm_max', 'nir_norm_mean', 'nir_norm_median', 'nir_norm_min', 'nir_norm_std', 'red_norm_max', 'red_norm_mean', 'red_norm_median', 'red_norm_min', 'red_norm_std', 's1_s2_month_gap', 'swir1_norm_max', 'swir1_norm_mean', 'swir1_norm_median', 'swir1_norm_min', 'swir1_norm_std', 'swir2_norm_max', 'swir2_norm_mean', 'swir2_norm_median', 'swir2_norm_min', 'swir2_norm_std', 'vh_frac_max', 'vh_frac_mean', 'vh_frac_median', 'vh_frac_min', 'vh_frac_std', 'vhvv_max', 'vhvv_mean', 'vhvv_median', 'vhvv_min', 'vhvv_std']


#  Retrain: SMALL, heavily regularized model on curated features


In [52]:
# ============================================================
#  Retrain: SMALL, heavily regularized model on curated features
# ============================================================
X7 = train_combined[curated_cols]
y7 = train_combined['label'].values
groups7 = train_combined['group_id'].values

gkf = GroupKFold(n_splits=5)
oof_proba7 = np.zeros(len(X7))

for tr_idx, va_idx in gkf.split(X7, y7, groups7):
    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.02,
        num_leaves=7,             # much shallower — was 31
        min_child_samples=60,     # forces broader splits, resists memorizing single locations
        reg_alpha=1.0,
        reg_lambda=1.0,
        subsample=0.7,
        colsample_bytree=0.6,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    )
    model.fit(X7.iloc[tr_idx], y7[tr_idx],
              eval_set=[(X7.iloc[va_idx], y7[va_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_proba7[va_idx] = model.predict_proba(X7.iloc[va_idx])[:, 1]

oof_pred7 = (oof_proba7 >= 0.5).astype(int)
f1_7 = f1_score(y7, oof_pred7)
auc_7 = roc_auc_score(y7, oof_proba7)
print(f"OOF F1:  {f1_7:.4f}")
print(f"OOF AUC: {auc_7:.4f}")
print(f"Blended: {0.6*f1_7 + 0.4*auc_7:.4f}")
print("\n(Expect this to be LOWER than 0.978 — that's the point. We want a number")
print(" that's honest, not inflated by memorization.)")

OOF F1:  0.9468
OOF AUC: 0.9865
Blended: 0.9627

(Expect this to be LOWER than 0.978 — that's the point. We want a number
 that's honest, not inflated by memorization.)


#  Submit the regularized, curated-feature model


In [53]:
# ============================================================
#  Submit the regularized, curated-feature model
# ============================================================
final_model_v2 = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.02, num_leaves=7,
    min_child_samples=60, reg_alpha=1.0, reg_lambda=1.0,
    subsample=0.7, colsample_bytree=0.6,
    class_weight='balanced', random_state=42, verbose=-1
)
final_model_v2.fit(X7, y7)

test_proba_v2 = final_model_v2.predict_proba(test_combined[curated_cols])[:, 1]
test_pred_v2 = (test_proba_v2 >= 0.5).astype(int)

submission_v2 = pd.DataFrame({
    'ID': test['ID'].values,
    'TargetF1': test_pred_v2,
    'TargetRAUC': test_proba_v2
})
print(submission_v2.head())
print("\nPredicted positive rate:", submission_v2['TargetF1'].mean())
submission_v2.to_csv('/content/drive/MyDrive/submission_v2.csv', index=False)
print("Saved submission_v2.csv")

                   ID  TargetF1  TargetRAUC
0  ID_TS_NEW_SBZAYD5I         1    0.909631
1  ID_TS_NEW_7SPRN3PB         1    0.967950
2  ID_TS_NEW_LZOWPHLC         1    0.802177
3  ID_TS_NEW_DN6TUF64         0    0.382584
4  ID_TS_NEW_95N82M49         0    0.166255

Predicted positive rate: 0.47378640776699027
Saved submission_v2.csv


# Controlled comparison: full features (177), varying regularization + class weighting


In [56]:
# ============================================================
# Controlled comparison: full features (177), varying regularization + class weighting
# ============================================================
def run_cv(X, y, groups, params, sample_weight=None):
    gkf = GroupKFold(n_splits=5)
    oof = np.zeros(len(X))
    for tr_idx, va_idx in gkf.split(X, y, groups):
        sw = sample_weight[tr_idx] if sample_weight is not None else None
        m = lgb.LGBMClassifier(**params)
        m.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=sw,
              eval_set=[(X.iloc[va_idx], y[va_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va_idx] = m.predict_proba(X.iloc[va_idx])[:, 1]
    f1 = f1_score(y, (oof>=0.5).astype(int))
    auc = roc_auc_score(y, oof)
    pos_rate = (oof>=0.5).mean()
    return f1, auc, 0.6*f1+0.4*auc, pos_rate, oof

X_full = train_combined[feature_cols_v3]
y_full = train_combined['label'].values
groups_full = train_combined['group_id'].values

configs = {
    'v1_baseline (num_leaves=31, balanced)': dict(
        n_estimators=2000, learning_rate=0.03, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=42, verbose=-1),
    'v1_no_balance': dict(
        n_estimators=2000, learning_rate=0.03, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1),
    'moderate_reg (num_leaves=15)': dict(
        n_estimators=2000, learning_rate=0.03, num_leaves=15,
        min_child_samples=20, reg_alpha=0.3, reg_lambda=0.3,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=42, verbose=-1),
    'deeper (num_leaves=63)': dict(
        n_estimators=2000, learning_rate=0.02, num_leaves=63,
        subsample=0.8, colsample_bytree=0.7,
        class_weight='balanced', random_state=42, verbose=-1),
}

results = {}
for name, params in configs.items():
    f1, auc, blend, pos_rate, oof = run_cv(X_full, y_full, groups_full, params)
    results[name] = (f1, auc, blend, pos_rate)
    print(f"{name:40s} F1={f1:.4f} AUC={auc:.4f} Blend={blend:.4f} pred_pos_rate={pos_rate:.3f}")

v1_baseline (num_leaves=31, balanced)    F1=0.9680 AUC=0.9936 Blend=0.9782 pred_pos_rate=0.402
v1_no_balance                            F1=0.9682 AUC=0.9935 Blend=0.9783 pred_pos_rate=0.399
moderate_reg (num_leaves=15)             F1=0.9684 AUC=0.9942 Blend=0.9787 pred_pos_rate=0.402
deeper (num_leaves=63)                   F1=0.9681 AUC=0.9937 Blend=0.9783 pred_pos_rate=0.402


In [58]:
!pip install catboost -q
import catboost as cb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.5 MB/s eta 0:00:00


In [59]:
import xgboost as xgb

In [60]:
def run_cv_generic(model_fn, X, y, groups):
    gkf = GroupKFold(n_splits=5)
    oof = np.zeros(len(X))

    # Check what type of classifier the lambda function builds
    sample_model = model_fn()
    is_lgb = "LGBMClassifier" in type(sample_model).__name__

    for tr_idx, va_idx in gkf.split(X, y, groups):
        m = model_fn()

        if is_lgb:
            # Modern LightGBM API silences logs using callbacks instead of verbose=False
            m.fit(
                X.iloc[tr_idx], y[tr_idx],
                eval_set=[(X.iloc[va_idx], y[va_idx])],
                callbacks=[]
            )
        else:
            # CatBoost and XGBoost still accept the traditional verbose flag in fit()
            m.fit(
                X.iloc[tr_idx], y[tr_idx],
                eval_set=[(X.iloc[va_idx], y[va_idx])],
                verbose=False
            )
        oof[va_idx] = m.predict_proba(X.iloc[va_idx])[:, 1]

    f1 = f1_score(y, (oof>=0.5).astype(int))
    auc = roc_auc_score(y, oof)
    return f1, auc, 0.6*f1+0.4*auc, oof

# LightGBM (best config from comparison — moderate_reg)
lgb_fn = lambda: lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.03, num_leaves=15,
    min_child_samples=20, reg_alpha=0.3, reg_lambda=0.3,
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced', random_state=42, verbose=-1,
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])

# CatBoost — different tree-growing strategy (symmetric/oblivious trees)
cat_fn = lambda: cb.CatBoostClassifier(
    iterations=2000, learning_rate=0.03, depth=6,
    l2_leaf_reg=5, auto_class_weights='Balanced',
    random_seed=42, verbose=False, early_stopping_rounds=100)

# XGBoost — different regularization defaults, histogram-based splits
xgb_fn = lambda: xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.03, max_depth=5,
    reg_alpha=0.3, reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_full==0).sum()/(y_full==1).sum(),
    eval_metric='auc', early_stopping_rounds=100,
    random_state=42, verbosity=0)

print("Training LightGBM...")
f1_l, auc_l, blend_l, oof_l = run_cv_generic(lgb_fn, X_full, y_full, groups_full)
print(f"LightGBM  F1={f1_l:.4f} AUC={auc_l:.4f} Blend={blend_l:.4f}")

print("Training CatBoost...")
f1_c, auc_c, blend_c, oof_c = run_cv_generic(cat_fn, X_full, y_full, groups_full)
print(f"CatBoost  F1={f1_c:.4f} AUC={auc_c:.4f} Blend={blend_c:.4f}")

print("Training XGBoost...")
f1_x, auc_x, blend_x, oof_x = run_cv_generic(xgb_fn, X_full, y_full, groups_full)
print(f"XGBoost   F1={f1_x:.4f} AUC={auc_x:.4f} Blend={blend_x:.4f}")

Training LightGBM...
LightGBM  F1=0.9705 AUC=0.9947 Blend=0.9802
Training CatBoost...
CatBoost  F1=0.9666 AUC=0.9941 Blend=0.9776
Training XGBoost...
XGBoost   F1=0.9693 AUC=0.9945 Blend=0.9794


#  Ensemble: average the 3 OOF probability sets, check if it beats each alone


In [61]:
# ============================================================
#  Ensemble: average the 3 OOF probability sets, check if it beats each alone
# ============================================================
oof_ensemble = (oof_l + oof_c + oof_x) / 3
f1_e = f1_score(y_full, (oof_ensemble>=0.5).astype(int))
auc_e = roc_auc_score(y_full, oof_ensemble)
print(f"ENSEMBLE  F1={f1_e:.4f} AUC={auc_e:.4f} Blend={0.6*f1_e+0.4*auc_e:.4f}")

# correlation between models' OOF predictions -- low correlation = genuine diversity, high = redundant
corr = pd.DataFrame({'lgb': oof_l, 'cat': oof_c, 'xgb': oof_x}).corr()
print("\nOOF prediction correlation between models:")
print(corr)


ENSEMBLE  F1=0.9696 AUC=0.9945 Blend=0.9796

OOF prediction correlation between models:
          lgb       cat       xgb
lgb  1.000000  0.993119  0.997742
cat  0.993119  1.000000  0.995607
xgb  0.997742  0.995607  1.000000


#  Build an honest internal "fake leaderboard" (pseudo-holdout)


In [62]:
# ============================================================
#  Build an honest internal "fake leaderboard" (pseudo-holdout)
# ============================================================
from sklearn.model_selection import train_test_split

pseudo_train_ids, pseudo_test_ids = train_test_split(
    train['ID'].values, test_size=0.2, random_state=42,
    stratify=train['label'].values
)
print(f"Pseudo-train locations: {len(pseudo_train_ids)}  Pseudo-test locations: {len(pseudo_test_ids)}")

train_subset = train[train['ID'].isin(pseudo_train_ids)].reset_index(drop=True)
holdout_subset = train[train['ID'].isin(pseudo_test_ids)].reset_index(drop=True)

# Training portion: augment as usual (15x), same pipeline as before
holdout_train_wide = build_masked_train(train_subset, n_aug=15, seed=42)
holdout_train_idx  = add_indices(holdout_train_wide)
holdout_train_agg  = aggregate_features(holdout_train_idx)
holdout_train_norm = aggregate_norm(add_normalized_bands(holdout_train_idx))
holdout_train_full = pd.concat([holdout_train_agg.reset_index(drop=True),
                                  holdout_train_norm.reset_index(drop=True)], axis=1)
holdout_train_full['label']    = holdout_train_wide['label'].values
holdout_train_full['group_id'] = holdout_train_wide['group_id'].values

# "Fake test" portion: SINGLE masked draw per location, mimicking real test exactly, never augmented
fake_test_wide = build_masked_train(holdout_subset, n_aug=1, seed=999)
fake_test_idx  = add_indices(fake_test_wide)
fake_test_agg  = aggregate_features(fake_test_idx)
fake_test_norm = aggregate_norm(add_normalized_bands(fake_test_idx))
fake_test_full = pd.concat([fake_test_agg.reset_index(drop=True),
                              fake_test_norm.reset_index(drop=True)], axis=1)
fake_test_full['label'] = fake_test_wide['label'].values

print("holdout_train_full:", holdout_train_full.shape)
print("fake_test_full:", fake_test_full.shape)

Pseudo-train locations: 1456  Pseudo-test locations: 365


/tmp/ipykernel_1318/3174094469.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aug['sim_window_len'] = lengths
/tmp/ipykernel_1318/3174094469.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aug['sim_window_start'] = starts


holdout_train_full: (21840, 184)
fake_test_full: (365, 183)


/tmp/ipykernel_1318/3174094469.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aug['sim_window_len'] = lengths
/tmp/ipykernel_1318/3174094469.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aug['sim_window_start'] = starts


#  Train on pseudo-train, score on fake-test — this is our real proxy for LB


In [63]:
# ============================================================
#  Train on pseudo-train, score on fake-test — this is our real proxy for LB
# ============================================================
feat_cols_holdout = [c for c in holdout_train_full.columns if c not in ['label','group_id']]

X_ht = holdout_train_full[feat_cols_holdout]
y_ht = holdout_train_full['label'].values

X_ft = fake_test_full[feat_cols_holdout]
y_ft = fake_test_full['label'].values

model_proxy = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.03, num_leaves=15,
    min_child_samples=20, reg_alpha=0.3, reg_lambda=0.3,
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced', random_state=42, verbose=-1
)
model_proxy.fit(X_ht, y_ht)
proxy_proba = model_proxy.predict_proba(X_ft)[:, 1]
proxy_pred = (proxy_proba >= 0.5).astype(int)

proxy_f1 = f1_score(y_ft, proxy_pred)
proxy_auc = roc_auc_score(y_ft, proxy_proba)
print(f"PROXY 'fake LB' score  F1={proxy_f1:.4f}  AUC={proxy_auc:.4f}  Blend={0.6*proxy_f1+0.4*proxy_auc:.4f}")
print(f"(compare: real LB was F1=0.803 AUC=0.844 Blend=0.819 for the equivalent full-feature model)")

# also report in-sample GroupKFold OOF on just this subset, for direct before/after comparison
gkf = GroupKFold(n_splits=5)
oof_proxy = np.zeros(len(X_ht))
groups_ht = holdout_train_full['group_id'].values
for tr_idx, va_idx in gkf.split(X_ht, y_ht, groups_ht):
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=15,
                            min_child_samples=20, reg_alpha=0.3, reg_lambda=0.3,
                            subsample=0.8, colsample_bytree=0.8,
                            class_weight='balanced', random_state=42, verbose=-1)
    m.fit(X_ht.iloc[tr_idx], y_ht[tr_idx], eval_set=[(X_ht.iloc[va_idx], y_ht[va_idx])],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_proxy[va_idx] = m.predict_proba(X_ht.iloc[va_idx])[:, 1]
oof_proxy_f1 = f1_score(y_ht, (oof_proxy>=0.5).astype(int))
oof_proxy_auc = roc_auc_score(y_ht, oof_proxy)
print(f"\nGroupKFold OOF on pseudo-train (for comparison): F1={oof_proxy_f1:.4f} AUC={oof_proxy_auc:.4f} Blend={0.6*oof_proxy_f1+0.4*oof_proxy_auc:.4f}")
print("Gap (OOF - proxy fake-LB):", (0.6*oof_proxy_f1+0.4*oof_proxy_auc) - (0.6*proxy_f1+0.4*proxy_auc))

PROXY 'fake LB' score  F1=0.9653  AUC=0.9971  Blend=0.9780
(compare: real LB was F1=0.803 AUC=0.844 Blend=0.819 for the equivalent full-feature model)

GroupKFold OOF on pseudo-train (for comparison): F1=0.9670 AUC=0.9949 Blend=0.9781
Gap (OOF - proxy fake-LB): 0.00014416515870163682


#  Add calibration-drift noise to augmentation (robustness to unseen shift)


In [64]:
# ============================================================
#  Add calibration-drift noise to augmentation (robustness to unseen shift)
# ============================================================
def build_masked_train_with_noise(train_df, n_aug=15, seed=42, noise_std=0.08):
    aug = build_masked_train(train_df, n_aug=n_aug, seed=seed)  # existing window-masking
    rng = np.random.default_rng(seed + 1)
    n = len(aug)

    # Per-row, per-band multiplicative gain drift (simulates calibration shift between acquisitions)
    for b in s2_bands:  # optical bands: multiplicative drift is physically realistic
        cols = [f'{b}_{m}' for m in months]
        gain = rng.normal(1.0, noise_std, size=n)  # e.g. ~8% row-level brightness variation
        for c in cols:
            valid = aug[c] != NA_VAL
            aug.loc[valid, c] = aug.loc[valid, c] * gain[valid.values]

    for b in s1_bands:  # SAR in dB: additive drift is more physically realistic
        cols = [f'{b}_{m}' for m in months]
        shift = rng.normal(0.0, noise_std * 3, size=n)  # dB shift
        for c in cols:
            valid = aug[c] != NA_VAL
            aug.loc[valid, c] = aug.loc[valid, c] + shift[valid.values]

    return aug

train_noisy_wide = build_masked_train_with_noise(train, n_aug=15, seed=42, noise_std=0.08)
train_noisy_idx  = add_indices(train_noisy_wide)
train_noisy_agg  = aggregate_features(train_noisy_idx)
train_noisy_norm = aggregate_norm(add_normalized_bands(train_noisy_idx))
train_noisy_full = pd.concat([train_noisy_agg.reset_index(drop=True),
                                train_noisy_norm.reset_index(drop=True)], axis=1)
train_noisy_full['label']    = train_noisy_wide['label'].values
train_noisy_full['group_id'] = train_noisy_wide['group_id'].values

print("Noisy augmented train:", train_noisy_full.shape)

feat_cols_noisy = [c for c in train_noisy_full.columns if c not in ['label','group_id']]
X_n = train_noisy_full[feat_cols_noisy]
y_n = train_noisy_full['label'].values
groups_n = train_noisy_full['group_id'].values

gkf = GroupKFold(n_splits=5)
oof_n = np.zeros(len(X_n))
for tr_idx, va_idx in gkf.split(X_n, y_n, groups_n):
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=15,
                            min_child_samples=20, reg_alpha=0.3, reg_lambda=0.3,
                            subsample=0.8, colsample_bytree=0.8,
                            class_weight='balanced', random_state=42, verbose=-1)
    m.fit(X_n.iloc[tr_idx], y_n[tr_idx], eval_set=[(X_n.iloc[va_idx], y_n[va_idx])],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_n[va_idx] = m.predict_proba(X_n.iloc[va_idx])[:, 1]

f1_n = f1_score(y_n, (oof_n>=0.5).astype(int))
auc_n = roc_auc_score(y_n, oof_n)
print(f"Noise-augmented OOF: F1={f1_n:.4f} AUC={auc_n:.4f} Blend={0.6*f1_n+0.4*auc_n:.4f}")

/tmp/ipykernel_1318/3174094469.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aug['sim_window_len'] = lengths
/tmp/ipykernel_1318/3174094469.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aug['sim_window_start'] = starts
/tmp/ipykernel_1318/1969301435.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1667.21875244 1590.95898741 1915.26413882 ... 1792.18381385 1929.30334158
 1830.94051123]' has dtype incompatible with int64, please 

Noisy augmented train: (27315, 184)
Noise-augmented OOF: F1=0.9564 AUC=0.9928 Blend=0.9709


#  Pseudo-labeling: train on train+noise, predict real test, add confident pseudo-labels


In [65]:
# ============================================================
#  Pseudo-labeling: train on train+noise, predict real test, add confident pseudo-labels
# ============================================================
# Step 1: train best-so-far model on ALL noisy-augmented train
pseudo_model = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.03, num_leaves=15,
    min_child_samples=20, reg_alpha=0.3, reg_lambda=0.3,
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced', random_state=42, verbose=-1
)
pseudo_model.fit(X_n, y_n)

test_full_for_pseudo = pd.concat([test_agg.reset_index(drop=True),
                                    test_norm_agg.reset_index(drop=True)], axis=1)[feat_cols_noisy]
test_proba_pseudo = pseudo_model.predict_proba(test_full_for_pseudo)[:, 1]

print("Test probability distribution:")
print(pd.Series(test_proba_pseudo).describe())

# Take high-confidence predictions only
HIGH, LOW = 0.90, 0.10
confident_mask = (test_proba_pseudo >= HIGH) | (test_proba_pseudo <= LOW)
print(f"\nConfident pseudo-labels: {confident_mask.sum()} / {len(test_proba_pseudo)} "
      f"({confident_mask.mean()*100:.1f}%)")
print(f"Of those, predicted positive: {(test_proba_pseudo[confident_mask]>=HIGH).sum()}, "
      f"predicted negative: {(test_proba_pseudo[confident_mask]<=LOW).sum()}")

pseudo_X = test_full_for_pseudo[confident_mask].copy()
pseudo_y = (test_proba_pseudo[confident_mask] >= HIGH).astype(int)
pseudo_weight = np.full(confident_mask.sum(), 0.5)  # downweight vs real labels

# Combine with real (noisy-augmented) training data
X_combined_pseudo = pd.concat([X_n, pseudo_X], axis=0).reset_index(drop=True)
y_combined_pseudo = np.concatenate([y_n, pseudo_y])
w_combined_pseudo = np.concatenate([np.ones(len(y_n)), pseudo_weight])

final_pseudo_model = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.03, num_leaves=15,
    min_child_samples=20, reg_alpha=0.3, reg_lambda=0.3,
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced', random_state=42, verbose=-1
)
final_pseudo_model.fit(X_combined_pseudo, y_combined_pseudo, sample_weight=w_combined_pseudo)

final_test_proba = final_pseudo_model.predict_proba(test_full_for_pseudo)[:, 1]
final_test_pred = (final_test_proba >= 0.5).astype(int)

submission_v3 = pd.DataFrame({
    'ID': test['ID'].values,
    'TargetF1': final_test_pred,
    'TargetRAUC': final_test_proba
})
print("\nFinal predicted positive rate:", submission_v3['TargetF1'].mean())
submission_v3.to_csv('/content/drive/MyDrive/submission_v3.csv', index=False)
print("Saved submission_v3.csv")

Test probability distribution:
count    1.030000e+03
mean     6.091331e-01
std      4.593935e-01
min      2.217302e-07
25%      6.088683e-04
50%      9.629919e-01
75%      9.982644e-01
max      9.999892e-01
dtype: float64

Confident pseudo-labels: 917 / 1030 (89.0%)
Of those, predicted positive: 564, predicted negative: 353

Final predicted positive rate: 0.6271844660194175
Saved submission_v3.csv
